# Data PRODUCER

This notebook has **only one job**: to drop new JSON files into the Volume,
acting like a real app, a sensor, or an external system sending events.

Its sole mission is to produce data.

---
## 0. Configuration — Same Volume as the Consumer

These paths **must be identical** to the ones in the Consumer notebook.
If you change one, change it in both.

In [0]:
catalog = "main"
schema  = "streaming_live"
volume  = "raw_data"

base_path   = f"/Volumes/{catalog}/{schema}/{volume}"
source_path = f"{base_path}/source"      # <- the "bucket": new files drop here

print("Bucket (source):", source_path)

Bucket (source): /Volumes/main/streaming_live/raw_data/source


In [0]:
# Create catalog / schema / volume if they do not exist, and the source folder
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {catalog}.{schema}.{volume}")
dbutils.fs.mkdirs(source_path)
print("Structure ready. Source folder created.")

Structure ready. Source folder created.


---
## 1. The Event Generator

Each call to `generar_batch()` writes **one JSON file** with several new orders.
For the streaming engine, **each new file = new data to process**.

In [0]:
import time, json, random
from datetime import datetime, timedelta

PRODUCTS = ["laptop", "keyboard", "monitor", "mouse", "headphones", "webcam"]
REGIONS  = ["north", "south", "east", "west", "central"]

# Writes a JSON Lines file with num_records new orders into the bucket.
def generate_batch(batch_id, num_records=5):
    orders = []
    base_time = datetime.now()

    for i in range(num_records):
        orders.append({
            "order_id":  f"ORD-{batch_id:04d}-{i:03d}",
            "product":   random.choice(PRODUCTS),
            "region":    random.choice(REGIONS),
            "quantity":  random.randint(1, 5),
            "price":     round(random.uniform(20.0, 1500.0), 2),
            "timestamp": (base_time + timedelta(seconds=i)).isoformat()
        })

    content = "\n".join(json.dumps(order) for order in orders)
    file_path = f"{source_path}/batch_{batch_id:04d}.json"

    dbutils.fs.put(file_path, content, overwrite=True)

    print(f"  -> batch_{batch_id:04d}.json  ({num_records} orders)")

print("Generator ready.")

Generator ready.


---
## 2. Manual Mode — One batch whenever you want

Execute this cell **every time you want \"new data to arrive\"** in the bucket.

Increment `BATCH_ID` (or increase it manually) on each execution so you don't overwrite the previous file.

In [0]:
# Increment this number on each execution: 1, 2, 3, ...
BATCH_ID = 1

generate_batch(BATCH_ID, num_records=random.randint(4, 8))
print(f"\nReady. Now go to the Consumer and run process_new().")

Wrote 734 bytes.
  -> batch_0003.json  (5 orders)

Ready. Now go to the Consumer and run process_new().


---
## 3. Reset

Deletes **only the files in the bucket**. It does not touch the Consumer's output.

In [0]:
#dbutils.fs.rm(source_path, True)
#dbutils.fs.mkdirs(source_path)
#print("Bucket cleared.")